# 06 Evaluation

**Owner.** Jack (Product Lead & Knowledge Architect) · **Supports.** Tim (F1 runs, human ratings for F4)

**Goal.** Measure how good the agent is, so that every claim in the presentation rests on a number:
answer quality, citation accuracy, cost and latency.

**Definition of Done.** 10 test cases by 18 Oct, 20 by 1 Nov. Judge runs reproducibly. Results for F1 and F4
as table and chart (tasks T09, T18, T20, T21).

## 1. Setup

In [ ]:
import os, json, re, time, pathlib
from dotenv import load_dotenv

load_dotenv()
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
SWISSAI_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"
CANDIDATE_MODELS = ["azure_ai/gpt-oss-120b", "gpt-5.6-luna"]   # add Apertus once notebook 01 found its name
JUDGE_MODEL = "azure_ai/mistral-large-3"   # different model family than the candidates, to limit self-preference

DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 2. The test set

One JSON line per case. `gold_articles` is what Hit@k and citation recall are measured against. `expected_answer`
is what the judge compares with. Every case must be checkable in the law, and the source of the case must be
noted (own wording, textbook, published decision) so we know we may use it.

In [ ]:
TESTSET_FILE = DATA_DIR / "testset.jsonl"

EXAMPLES = [
    {"id": "AR-01", "area": "arbeitsrecht", "difficulty": "easy", "source": "own wording",
     "question": "Wie lange ist die Kündigungsfrist im dritten Dienstjahr, wenn nichts vereinbart ist?",
     "expected_answer": "Zwei Monate auf das Ende eines Monats.",
     "gold_articles": ["OR-335c"]},
    {"id": "AR-02", "area": "arbeitsrecht", "difficulty": "medium", "source": "own wording",
     "question": "Ich bin im dritten Dienstjahr und seit zwei Wochen krank. Mein Chef kündigt mir. Ist das gültig?",
     "expected_answer": "Nein. Im zweiten bis fünften Dienstjahr gilt bei Krankheit eine Sperrfrist von 90 Tagen, "
                        "eine Kündigung in dieser Zeit ist nichtig.",
     "gold_articles": ["OR-336c"]},
    {"id": "MR-01", "area": "mietrecht", "difficulty": "medium", "source": "own wording",
     "question": "Ich habe zwei Monatsmieten nicht bezahlt. Was kann der Vermieter tun?",
     "expected_answer": "Er kann schriftlich eine Zahlungsfrist von mindestens 30 Tagen mit Kündigungsandrohung "
                        "setzen und danach mit mindestens 30 Tagen auf Ende eines Monats kündigen.",
     "gold_articles": ["OR-257d"]},
]

if not TESTSET_FILE.exists():
    TESTSET_FILE.write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in EXAMPLES), encoding="utf-8")

testset = [json.loads(line) for line in TESTSET_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
print(len(testset), "cases")

## 3. Deterministic metrics

These need no model and are therefore cheap and fully reproducible.

- **Citation precision:** share of cited articles that are correct (in the gold list).
- **Citation recall:** share of gold articles the answer actually cites.
- **Latency and tokens:** measured per call.

In [ ]:
CITATION = re.compile(r"Art\.\s*(\d+[a-z]*)\s*(?:Abs\.\s*\d+\s*)?(?:lit\.\s*[a-z]\s*)?(OR|ZGB)\b")

def cited_articles(answer):
    return {f"{law}-{art.lower()}".upper() for art, law in CITATION.findall(answer)}

def citation_scores(answer, gold_articles):
    cited, gold = cited_articles(answer), {g.upper() for g in gold_articles}
    precision = len(cited & gold) / len(cited) if cited else 0.0
    recall = len(cited & gold) / len(gold) if gold else 0.0
    return {"citation_precision": round(precision, 2), "citation_recall": round(recall, 2)}

assert citation_scores("Gemäss Art. 336c Abs. 1 OR ist sie nichtig.", ["OR-336c"]) == \
       {"citation_precision": 1.0, "citation_recall": 1.0}
assert citation_scores("Art. 336c OR und Art. 999 OR", ["OR-336c"])["citation_precision"] == 0.5

## 4. LLM-as-a-judge

A single 1 to 10 score is hard to reproduce. We split the rubric into yes/no criteria and let the judge answer each
one. The score is the share of criteria met, from 0 to 100. A judge model plays the same role as a reward model in
RLHF (lecture 12), so it can have the same weaknesses: it may prefer long or confident answers. Experiment F4
checks it against our own ratings.

In [ ]:
CRITERIA = {
    "correct_norm": "Die Antwort stützt sich auf die richtige Gesetzesbestimmung.",
    "correct_result": "Das Ergebnis stimmt inhaltlich mit der Musterlösung überein.",
    "no_invented_sources": "Die Antwort nennt keine erfundenen Artikel oder Entscheide.",
    "complete": "Wesentliche Voraussetzungen und Fristen aus der Musterlösung sind erwähnt.",
    "clear_structure": "Die Antwort ist klar gegliedert (Norm, Subsumtion, Ergebnis).",
}

JUDGE_PROMPT = (
    "Du bewertest die Antwort eines juristischen Assistenten. Beurteile jedes Kriterium mit true oder false.\n"
    "Antworte nur mit JSON im Format {{\"criteria\": {{<name>: true|false}}, \"comment\": \"<ein Satz>\"}}.\n\n"
    "Kriterien:\n{criteria}\n\nFrage: {question}\n\nMusterlösung: {expected}\n\nAntwort: {answer}"
)

def parse_judge(text):
    """Extract the JSON object from the judge reply, robust to extra text around it."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    data = json.loads(match.group(0)) if match else {}
    criteria = {name: bool(data.get("criteria", {}).get(name, False)) for name in CRITERIA}
    return {"criteria": criteria, "score": round(100 * sum(criteria.values()) / len(criteria)),
            "comment": data.get("comment", "")}

example = parse_judge('Hier: {"criteria": {"correct_norm": true, "correct_result": true, "no_invented_sources": true, '
                      '"complete": false, "clear_structure": true}, "comment": "Frist fehlt."}')
assert example["score"] == 80

In [ ]:
from langchain_openai import ChatOpenAI

def make_llm(model, base_url=LLM_BASE_URL, key_env="LITELLM_API_KEY"):
    return ChatOpenAI(model=model, base_url=base_url, api_key=os.environ[key_env], temperature=0)

judge = make_llm(JUDGE_MODEL)

def judge_answer(case, answer):
    criteria_text = "\n".join(f"- {name}: {text}" for name, text in CRITERIA.items())
    reply = judge.invoke(JUDGE_PROMPT.format(
        criteria=criteria_text, question=case["question"], expected=case["expected_answer"], answer=answer,
    )).content
    return parse_judge(reply)

## 5. Experiment F1: model comparison

For now the candidate answers with the prompt only (baseline). As soon as the graph from notebook 04 runs,
`answer_fn` calls the agent instead, and the same loop produces the numbers for F1 and F2.

Prices per million tokens are filled in from the LiteLLM proxy or the provider. Without a price the cost column stays empty.

In [ ]:
import pandas as pd

PRICE_PER_1M_TOKENS = {   # (input, output) in USD, fill in
    "azure_ai/gpt-oss-120b": (None, None),
    "gpt-5.6-luna": (None, None),
}

def baseline_answer(model, question):
    llm = make_llm(model)
    start = time.perf_counter()
    reply = llm.invoke("Beantworte den Fall nach Schweizer Recht und zitiere die Artikel als 'Art. X OR'.\n\n"
                       + question)
    usage = reply.usage_metadata or {}
    return reply.content, time.perf_counter() - start, usage.get("input_tokens", 0), usage.get("output_tokens", 0)

def cost(model, tokens_in, tokens_out):
    price_in, price_out = PRICE_PER_1M_TOKENS.get(model, (None, None))
    if price_in is None:
        return None
    return round((tokens_in * price_in + tokens_out * price_out) / 1_000_000, 5)

def run_experiment(models, cases, answer_fn=baseline_answer):
    rows = []
    for model in models:
        for case in cases:
            answer, seconds, t_in, t_out = answer_fn(model, case["question"])
            verdict = judge_answer(case, answer)
            rows.append({"model": model, "case": case["id"], "judge_score": verdict["score"],
                         **citation_scores(answer, case["gold_articles"]),
                         "seconds": round(seconds, 1), "cost_usd": cost(model, t_in, t_out),
                         **{f"c_{k}": v for k, v in verdict["criteria"].items()}, "answer": answer})
    return pd.DataFrame(rows)

f1 = run_experiment(CANDIDATE_MODELS, testset)
summary = f1.groupby("model")[["judge_score", "citation_precision", "citation_recall", "seconds"]].mean().round(2)
summary

In [ ]:
import matplotlib.pyplot as plt

ax = summary[["judge_score"]].plot.bar(legend=False, figsize=(6, 3.5), rot=0)
ax.set_ylabel("Judge score (0 to 100)")
ax.set_title("F1: answer quality per model")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 6. Experiment F4: does the judge agree with us?

Jack and Tim each rate the same answers on the same yes/no criteria. We then compare with the judge.
Raw agreement overestimates, because two raters who always say "yes" agree by chance. Cohen's kappa corrects
for that: 0 means chance level, 1 means perfect agreement, above 0.6 counts as substantial.

In [ ]:
def cohens_kappa(a, b):
    """Cohen's kappa for two lists of booleans of equal length."""
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    p_a, p_b = sum(a) / n, sum(b) / n
    expected = p_a * p_b + (1 - p_a) * (1 - p_b)
    return 1.0 if expected == 1 else round((observed - expected) / (1 - expected), 3)

human = [True, True, False, True, False, True, True, False]
judge_ratings = [True, True, False, True, True, True, False, False]
print("agreement:", sum(h == j for h, j in zip(human, judge_ratings)) / len(human),
      "| kappa:", cohens_kappa(human, judge_ratings))
assert cohens_kappa([True, False, True], [True, False, True]) == 1.0

## Your turn

- Write five new test cases, at least two of them tenancy law, with gold articles and a short model answer.
- Let the judge rate the same answer three times. Is the score stable at `temperature=0`?
- Which criterion would a lawyer weight most? Should the score stay an unweighted mean?

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | Yes/no criteria instead of a 1 to 10 scale | One holistic score | More reproducible, and every point can be explained. |
| Oct 2026 | Judge from a different model family than the candidates | Same model judges itself | Limits self-preference bias. |
| | Final judge model: | | |
| | Test set size and sources: | | |

## Building block to take away

`citation_scores`, `parse_judge`, `judge_answer`, `run_experiment` and `cohens_kappa` move into the repository as
`eval/`. The test set stays in `data/testset.jsonl`.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>